# Aufruf-Graph-Analyse und Erkennung starker Zusammenhangskomponenten (SCC)

Dieses Notebook untersucht die erweiterten Code-Analysefunktionen von UnifyWeaver:

- **Aufruf-Graph-Konstruktion** - Aufbau von Abhängigkeitsgraphen aus Prolog-Code
- **SCC-Erkennung** - Auffinden starker Zusammenhangskomponenten (wechselseitige Rekursion)
- **Musteranalyse** - Rekursionsmuster verstehen
- **Abhängigkeitsvisualisierung** - Beziehungen zwischen Prädikaten visualisieren

## Lernziele

- Verstehen, wie UnifyWeaver die Codestruktur analysiert
- Aufruf-Graphen erstellen und untersuchen
- Wechselseitige Rekursion mit dem Tarjan-Algorithmus erkennen
- Code-Abhängigkeiten visualisieren

## Einrichtung

Laden von UnifyWeaver und Analysemodulen.

In [ ]:
% Initialisierung laden
['../init'].

% Analysemodule laden
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Beispiel 1: Einfacher Aufruf-Graph

Beginnen wir mit einem einfachen Prädikat und erstellen dessen Aufruf-Graphen.

In [ ]:
% Ancestor-Prädikat definieren
:- dynamic ancestor/2.
:- dynamic parent/2.

% Parent-Fakten
parent(abraham, isaac).
parent(isaac, jacob).

% Ancestor-Regeln
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### Aufruf-Graph erstellen

In [ ]:
% Aufruf-Graphen für ancestor erstellen
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Abhängigkeiten analysieren

In [ ]:
% Alle Abhängigkeiten von ancestor/2 abrufen
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Prüfen, ob selbst-rekursiv
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## Beispiel 2: Erkennung wechselseitiger Rekursion

Erkennen wir nun wechselseitige Rekursion anhand des Gerade/Ungerade-Beispiels.

In [ ]:
% Wechselseitig rekursive Prädikate definieren
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### Aufruf-Graph für beide Prädikate erstellen

In [ ]:
% Aufruf-Graphen für beide Prädikate erstellen
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Starke Zusammenhangskomponenten (SCCs) finden

In [ ]:
% Graph neu erstellen, da Variablen zwischen Notebook-Zellen nicht erhalten bleiben
build_call_graph([is_even/1, is_odd/1], _Graph),
% SCCs mit Tarjans Algorithmus finden
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### Prüfen, ob SCC trivial ist

In [ ]:
% Abgeleitete Werte neu berechnen, damit diese Zelle auch unabhängig läuft
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Jede SCC prüfen
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## Beispiel 3: Komplexer Aufruf-Graph

Analysieren wir ein komplexeres System mit mehreren Prädikaten.

In [ ]:
% Kleines Programm mit mehreren Prädikaten definieren
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent verwendet parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: gleicher Elternteil, verschiedene Kinder
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: Eltern sind Geschwister
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### Vollständigen Aufruf-Graphen erstellen

In [ ]:
% Aufruf-Graphen für alle Prädikate erstellen
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Prädikatgruppen finden

Finden der wechselseitig rekursiven Prädikatgruppe, die ein Startprädikat enthält.

In [ ]:
% Wechselseitig rekursive Gruppe finden, die cousin/2 enthält
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## Beispiel 4: Mustererkennung

Verwenden wir Musterabgleicher, um Rekursionstypen zu analysieren.

In [ ]:
% Verschiedene Rekursionsmuster definieren
:- dynamic count/3.     % Endrekursiv
:- dynamic factorial/2. % Linear rekursiv
:- dynamic fib/2.       % Baumrekursiv (oder linear, falls erkannt)

% Endrekursives Zählen
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Linear rekursive Fakultät
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Fibonacci (kann als linear oder baumrekursiv erkannt werden)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### Endrekursion erkennen

In [ ]:
% Prüfen, ob count/3 endrekursiv ist
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### Lineare Rekursion erkennen

In [ ]:
% Prüfen, ob factorial/2 linear rekursiv ist
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### Rekursive Aufrufe zählen

In [ ]:
% Rekursive Aufrufe in Fibonacci zählen
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## Visualisierung im DOT-Format

Erstellen wir eine Graphviz-DOT-Darstellung unseres Aufruf-Graphen.

In [ ]:
% Hilfsfunktion zur Erzeugung des DOT-Formats
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% DOT für Gerade/Ungerade-Graphen erzeugen
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### DOT-Datei speichern

In [ ]:
% DOT-Quellcode neu erstellen, da Variablen zwischen Zellen nicht erhalten bleiben
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## Übung: Eigenen Code analysieren

Versuchen Sie, eigene Prädikate zu definieren und zu analysieren!

In [ ]:
% Definieren Sie hier Ihre Prädikate
% Erstellen Sie dann Aufruf-Graphen, finden Sie SCCs und erkennen Sie Muster

% Beispiel:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## Zusammenfassung

In diesem Notebook haben Sie gelernt:

✅ Wie man Aufruf-Graphen aus Prolog-Code erstellt

✅ Wie man starke Zusammenhangskomponenten (SCCs) für wechselseitige Rekursion erkennt

✅ Wie man Musterabgleicher verwendet, um Rekursionstypen zu klassifizieren

✅ Wie man Prädikatabhängigkeiten analysiert

✅ Wie man Aufruf-Graphen im DOT-Format visualisiert

## Weiterführende Themen

Für tiefergehende Analysen:

- **Topologische Sortierung**: Verwenden Sie `topological_order/2`, um SCCs nach Abhängigkeiten zu ordnen
- **Benutzerdefinierte Musterabgleicher**: Schreiben Sie eigene Prädikate zur Mustererkennung
- **Akkumulatormuster-Extraktion**: Verwenden Sie `extract_accumulator_pattern/2` für detaillierte Analysen
- **Lineare Rekursion verbieten**: Verwenden Sie `forbid_linear_recursion/1`, um andere Kompilierungsstrategien zu erzwingen

## Referenzen und verwandte Dateien

- Kapitel 10: Prolog-Introspektion und Theorie
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`